# 解码、缓存、精度与训练目标的机制实验

围绕小矩阵与人工分数检查：top-p边界、KV缓存等价、量化误差、DPO/LoRA数值、候选选择与Pareto决策。每部分明确验证范围，没有GPU吞吐或真实模型质量结论。

[推理](../01-concepts/inference/README.md) · [训练](../01-concepts/training/README.md) · [推理模型](../01-concepts/reasoning/README.md) · [选型](../01-concepts/model-selection/README.md) · [源码](../05-code/model_mechanics.py)。

In [1]:
from pathlib import Path
import sys, platform
import numpy as np
here = Path.cwd().resolve()
repo = next(p for p in [here, *here.parents] if (p / '10-Knowledge').is_dir())
sys.path.insert(0, str(repo / '10-Knowledge' / '02-foundation-models' / '05-code'))
from model_mechanics import *
np.set_printoptions(precision=6, suppress=True)
print('Python:', platform.python_version(), 'NumPy:', np.__version__)
print('Data: synthetic teaching examples; no model download or API call')

Python: 3.12.13 NumPy: 2.5.2
Data: synthetic teaching examples; no model download or API call


## 1. Top-p必须保留跨过阈值的候选

概率[0.6,0.25,0.1,0.05]、p=0.8时保留前两个，再归一化。改变温度时比较分布，不用随机抽取一次的token代表总体行为。

In [2]:
prob=np.array([.6,.25,.1,.05]);logits=np.log(prob)
nucleus=decode_distribution(logits,top_p=.8)
print('top-p=.8:',nucleus)
assert np.allclose(nucleus,[.6/.85,.25/.85,0,0])
for temp in [.5,1.,2.]:
    print('temperature:',temp,'probabilities:',decode_distribution(logits,temperature=temp))
assert np.allclose(decode_distribution(logits,top_k=1),[1,0,0,0])
try:decode_distribution(logits,temperature=0)
except ValueError as error:print('zero temperature rejected; greedy uses argmax:',error)

top-p=.8: [0.705882 0.294118 0.       0.      ]
temperature: 0.5 probabilities: [0.827586 0.143678 0.022989 0.005747]
temperature: 1.0 probabilities: [0.6  0.25 0.1  0.05]
temperature: 2.0 probabilities: [0.426909 0.275568 0.174285 0.123238]
zero temperature rejected; greedy uses argmax: Use vector logits, temperature>0 and 0<top_p<=1


## 2. 单层因果计算与逐token缓存一致

固定参数、输入与mask，不使用Dropout。逐token缓存只计算新投影，复用历史K/V。这里没有FFN、位置编码和完整语言模型；比较的是相同单层数学计算。

In [3]:
rng=np.random.default_rng(12)
x=rng.normal(size=(8,6))
wq,wk,wv=[rng.normal(size=(6,4)) for _ in range(3)]
full,_=attention(x@wq,x@wk,x@wv,np.tril(np.ones((8,8),bool)))
cached=cached_attention(x,wq,wk,wv)
print('full shape:',full.shape,'cached shape:',cached.shape)
print('max error:',float(np.max(abs(full-cached))))
assert np.allclose(full,cached,atol=1e-12)
for heads in [32,8,1]:
    size=kv_cache_bytes(1,32,4096,heads,128,2)
    print('KV heads:',heads,'bytes:',size,'GiB:',size/2**30)
assert kv_cache_bytes(1,32,4096,8,128,2)==2**29

full shape: (8, 4) cached shape: (8, 4)
max error: 2.220446049250313e-15
KV heads: 32 bytes: 2147483648 GiB: 2.0
KV heads: 8 bytes: 536870912 GiB: 0.5
KV heads: 1 bytes: 67108864 GiB: 0.0625


## 3. 量化误差既要看权重，也要看输出

同一矩阵比较8比特与4比特的对称均匀量化。`int8`数组只是储存容器，4比特代码没有实际打包到半字节，因此不以NumPy数组大小宣称得到4比特内存收益。

In [4]:
weights=rng.normal(size=(6,4))
for bits in [8,4]:
    quantized,scale,restored=symmetric_quantize(weights,bits)
    weight_error=np.max(abs(restored-weights))
    output_error=np.max(abs(x@restored-x@weights))
    print('bits:',bits,'scale:',scale,'max weight error:',weight_error,'max output error:',output_error)
    assert weight_error<=scale/2+1e-12
q0,s0,r0=symmetric_quantize(np.zeros((2,2)))
assert np.all(q0==0) and np.all(r0==0)
near_tie=np.array([1.,.99999])
perturbed=near_tie+np.array([-.0001,.0001])
print('near tie argmax before/after:',int(near_tie.argmax()),int(perturbed.argmax()))
assert near_tie.argmax()!=perturbed.argmax()

bits: 8 scale: 0.020568235356056744 max weight error: 0.009635539109827285 max output error: 0.031436806335422285
bits: 4 scale: 0.3731665557456009 max weight error: 0.17239102417104246 max output error: 0.5905223243643467
near tie argmax before/after: 0 1


## 4. DPO损失的方向与LoRA等价

DPO用参考模型相对差值Δ，不是直接奖励答案长度。LoRA计算Wx+(α/r)BAx，与合并权重后的线性输出应该一致。该单元没有训练任何大模型。

In [5]:
for delta in [0.,1.,-1.]:
    beta=1.
    dpo_loss=np.logaddexp(0,-beta*delta)
    print('delta:',delta,'DPO loss:',float(dpo_loss))
W=rng.normal(size=(4,6));A=rng.normal(size=(2,6));B=rng.normal(size=(4,2));vector=rng.normal(size=6)
alpha,rank=4.,2
separate=W@vector+(alpha/rank)*(B@(A@vector))
merged=(W+(alpha/rank)*(B@A))@vector
print('LoRA merge max difference:',float(np.max(abs(separate-merged))))
assert np.allclose(separate,merged)
# Both factors zero means both gradients zero: dL/dB = g*(Ax)^T; dL/dA = (B^Tg)*x^T.
g=np.ones(4);zero_A=np.zeros_like(A);zero_B=np.zeros_like(B)
assert np.all(np.outer(g,zero_A@vector)==0)
assert np.all(np.outer(zero_B.T@g,vector)==0)
print('Both-zero LoRA factors have zero first-step gradients (verified)')

delta: 0.0 DPO loss: 0.6931471805599453
delta: 1.0 DPO loss: 0.31326168751822286
delta: -1.0 DPO loss: 1.3132616875182228
LoRA merge max difference: 3.552713678800501e-15
Both-zero LoRA factors have zero first-step gradients (verified)


## 5. 有正确候选与能选出正确候选是两件事

三条人工候选中有两条等价正确答案。用精确有理数验证；然后计算独立候选假设下至少一条成功的概率。真实模型候选往往相关，这只是概率演示。

In [6]:
from fractions import Fraction
candidates=['17','19','34/2']
valid=[answer for answer in candidates if Fraction(answer)==2*12-7]
print('exact verifier passed:',valid)
assert valid==['17','34/2']
p,k=.4,3
print('independent oracle probability of at least one success:',1-(1-p)**k)

exact verifier passed: ['17', '34/2']
independent oracle probability of at least one success: 0.784


## 6. 模型选择先删除被完全支配的配置

A/B/C是虚构模型，质量、成本、p95均为教学数据。程序只验证Pareto规则，不给真实推荐。

In [7]:
models=[dict(name='A',success=.80,cost=.2,p95=3.),dict(name='B',success=.90,cost=.4,p95=4.),dict(name='C',success=.78,cost=.3,p95=4.)]
def dominates(a,b):
    return (a['success']>=b['success'] and a['cost']<=b['cost'] and a['p95']<=b['p95'] and
            (a['success']>b['success'] or a['cost']<b['cost'] or a['p95']<b['p95']))
frontier=[m['name'] for m in models if not any(dominates(other,m) for other in models)]
print('Pareto frontier:',frontier)
assert frontier==['A','B']
feasible=[m['name'] for m in models if m['success']>=.88 and m['cost']<=.3]
print('success>=.88 and cost<=.3 -> feasible:',feasible)
assert feasible==[]

Pareto frontier: ['A', 'B']
success>=.88 and cost<=.3 -> feasible: []


## 观察与边界

已验证：采样过滤边界、缓存数值等价、KV容量公式、均匀量化误差、LoRA矩阵合并、DPO损失方向、精确候选检查和Pareto支配。未验证：真实LLM能力、GPU速度、生产缓存隔离、推测解码、真实模型微调与线上选型收益。

练习：让top-p阈值恰好等于0.6，检查边界；让一条候选不是合法分数，给验证器补类型错误处理；加入模型D后重新计算可行集与前沿。